In [1]:
import requests
import sqlite3
import random

def setup_database():
    """Initializes the SQLite database and creates the countries table."""
    conn = sqlite3.connect('world_data.db')
    cursor = conn.cursor()

    # Create table with the specified attributes
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS countries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            capital TEXT,
            flag TEXT,
            subregion TEXT,
            population INTEGER
        )
    ''')
    conn.commit()
    return conn

def fetch_and_store_random_countries():
    # 1. Fetch all countries from the REST Countries API
    api_url = "https://restcountries.com/v3.1/all"
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        all_countries = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return

    # 2. Select 10 random countries from the list
    random_countries = random.sample(all_countries, 10)

    # 3. Connect to database
    conn = setup_database()
    cursor = conn.cursor()

    # 4. Parse and Insert data
    for country in random_countries:
        # Extracting specific attributes with fallbacks for missing data
        name = country.get('name', {}).get('common', 'N/A')

        # Capital is returned as a list, we take the first element
        capitals = country.get('capital', ['N/A'])
        capital = capitals[0] if capitals else 'N/A'

        # 'flags' returns an object; we'll store the PNG URL
        flag = country.get('flags', {}).get('png', 'N/A')

        subregion = country.get('subregion', 'N/A')
        population = country.get('population', 0)

        cursor.execute('''
            INSERT INTO countries (name, capital, flag, subregion, population)
            VALUES (?, ?, ?, ?, ?)
        ''', (name, capital, flag, subregion, population))

    conn.commit()
    conn.close()
    print("Successfully added 10 random countries to the database.")

if __name__ == "__main__":
    fetch_and_store_random_countries()

Error fetching data: 400 Client Error: Bad Request for url: https://restcountries.com/v3.1/all
